# Tarea 5: De Máxima Verosimilitud Condicional a MCO

**Objetivo:** Demostrar formalmente cómo la estimación por Máxima Verosimilitud Condicional equivale a Mínimos Cuadrados Ordinarios (MCO) bajo supuestos gaussianos.

## 1. Demostración Matemática

Dado el modelo de regresión lineal:
$$y_t = \mathbf{x}_t' \boldsymbol{\beta} + \epsilon_t, \quad \epsilon_t \mid \mathbf{x}_t \sim \text{NID}(0, \sigma^2)$$

La **función de densidad condicional** de $y_t$ dado $\mathbf{x}_t$ es:
$$f(y_t \mid \mathbf{x}_t; \boldsymbol{\beta}, \sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(y_t - \mathbf{x}_t' \boldsymbol{\beta})^2}{2\sigma^2}\right)$$

La **función de Log-Verosimilitud Condicional** para la muestra completa es:
$$\ln L(\boldsymbol{\beta}, \sigma^2) = \sum_{t=1}^T \ln f(y_t \mid \mathbf{x}_t; \boldsymbol{\beta}, \sigma^2)$$
$$\ln L(\boldsymbol{\beta}, \sigma^2) = -\frac{T}{2}\ln(2\pi) - \frac{T}{2}\ln(\sigma^2) - \frac{1}{2\sigma^2} \sum_{t=1}^T (y_t - \mathbf{x}_t' \boldsymbol{\beta})^2$$

### Maximización respecto a $\boldsymbol{\beta}$:
Dado que los dos primeros términos no dependen de $\boldsymbol{\beta}$ y $-1/(2\sigma^2) < 0$, **maximizar $\ln L$ con respecto a $\boldsymbol{\beta}$ es equivalente a minimizar la Suma de Residuos al Cuadrado (SRC)**:
$$\max_{\boldsymbol{\beta}} \ln L(\boldsymbol{\beta}, \sigma^2) \iff \min_{\boldsymbol{\beta}} \sum_{t=1}^T (y_t - \mathbf{x}_t' \boldsymbol{\beta})^2$$

Derivando respecto a $\boldsymbol{\beta}$ e igualando a cero:
$$\frac{\partial \ln L}{\partial \boldsymbol{\beta}} = \frac{1}{\sigma^2} \mathbf{X}'(\mathbf{y} - \mathbf{X}\boldsymbol{\beta}) = \mathbf{0}$$
$$\mathbf{X}'\mathbf{y} - \mathbf{X}'\mathbf{X}\boldsymbol{\beta} = \mathbf{0} \implies \hat{\boldsymbol{\beta}}_{MV} = (\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{y} = \hat{\boldsymbol{\beta}}_{MCO}$$

**Conclusión:** Bajo errores normales i.i.d., el estimador de Máxima Verosimilitud Condicional $\hat{\boldsymbol{\beta}}_{MV}$ es exactamente idéntico al estimador MCO $\hat{\boldsymbol{\beta}}_{MCO}$.

In [1]:
import numpy as np

np.random.seed(2024)
T = 200
X = np.column_stack([np.ones(T), np.random.uniform(1, 10, T)])
beta_real = np.array([2.5, 1.8])
sigma_real = 1.2

epsilon = np.random.normal(0, sigma_real, T)
y = X @ beta_real + epsilon

# 1. Estimación MCO
beta_mco = np.linalg.inv(X.T @ X) @ X.T @ y

# 2. Estimación MV Condicional (por optimización numérica)
def neg_log_likelihood(params):
    b0, b1, sigma = params
    if sigma <= 0:
        return 1e10
    y_pred = b0*X[:, 0] + b1*X[:, 1]
    res = y - y_pred
    return 0.5*T*np.log(2*np.pi) + 0.5*T*np.log(sigma**2) + (1/(2*sigma**2))*np.sum(res**2)

import scipy.optimize as opt
res_mv = opt.minimize(neg_log_likelihood, x0=[1.0, 1.0, 1.0], method='L-BFGS-B')

print("=== COMPARACIÓN MCO VS MÁXIMA VEROSIMILITUD ===")
print(f"Beta_0: Verdadero = {beta_real[0]:.3f} | MCO = {beta_mco[0]:.4f} | MV = {res_mv.x[0]:.4f}")
print(f"Beta_1: Verdadero = {beta_real[1]:.3f} | MCO = {beta_mco[1]:.4f} | MV = {res_mv.x[1]:.4f}")


=== COMPARACIÓN MCO VS MÁXIMA VEROSIMILITUD ===
Beta_0: Verdadero = 2.500 | MCO = 2.5084 | MV = 2.5084
Beta_1: Verdadero = 1.800 | MCO = 1.8003 | MV = 1.8003
